Load Python Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, time, random, string, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag, word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

random.seed(42)
np.random.seed(42)
import sklearn, matplotlib


Load NLTK package

In [ ]:
packages = [
    "punkt", "stopwords", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "punkt_tab"
]
for p in packages:
    try:
        nltk.download(p, quiet=True)
    except Exception as e:
        print(f"NLTK download failed for {p}: {e}")

STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
wnl = WordNetLemmatizer()
print("NLTK done")

Load dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
from collections import Counter

df = pd.read_csv("fake.csv", nrows=40000, low_memory=False)

print(df.head(3))

print("Label distribution:", Counter(df["type"].tolist()))

print("Total rows:", len(df))


Tokenization

In [ ]:
from nltk.tokenize import word_tokenize

def tokenize(column_data):
    return [word_tokenize(str(t)) for t in column_data]

tokenized_text = tokenize(df['text'])

print(tokenized_text[0][:50])

Case folding




In [ ]:

def to_lower(tokens_list):
    return [[t.lower() for t in toks] for toks in tokens_list]

lowered_corpus = to_lower(tokenized_text)

print(lowered_corpus[0][:50])

Punctuation Removal

In [ ]:
import re
def clean_tokens(doc_tokens):
    clean_doc = []
    for token in doc_tokens:
        new_token = re.sub(r'[^a-zA-Z0-9]', '', token)

        if new_token != '':
            clean_doc.append(new_token)
    return clean_doc

final_cleaned_text = [clean_tokens(tokens) for tokens in lowered_corpus]

print(final_cleaned_text[0][:30])

Stop word removal

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens_list):
    return [word for word in tokens_list if word not in stop_words]

text_without_stopwords = [remove_stopwords(tokens) for tokens in final_cleaned_text]

print(text_without_stopwords[0][:30])

Lemmatization

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()

def lemmatize_text(token_list):
    return [lemmatizer.lemmatize(token) for token in token_list]

lemmatized_output = [lemmatize_text(tokens) for tokens in text_without_stopwords]

print(lemmatized_output[0][:30])

Synonym Substitution

In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def synonym_substitution(tokens_list):
    new_sentence = []
    for word in tokens_list:
        synonyms = wordnet.synsets(word)

        if synonyms:
            new_word = synonyms[0].lemmas()[0].name()

            if new_word != word and "_" not in new_word:
                new_sentence.append(new_word)
            else:
                new_sentence.append(word)
        else:
            new_sentence.append(word)

    return new_sentence

synonym_replaced_data = [synonym_substitution(tokens) for tokens in lemmatized_output]

print(synonym_replaced_data[0][:30])

Split (Train , Test/Validation)

In [ ]:
from sklearn.model_selection import train_test_split

X_text = [" ".join(tokens) for tokens in synonym_replaced_data]

X = X_text
y = df['type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Total Data:", len(X))
print("Training Data Size (70%):", len(X_train))
print("Testing Data Size (30%):", len(X_test))

Using TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=5)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Training Data Shape:", X_train_tfidf.shape)
print("Testing Data Shape:", X_test_tfidf.shape)

Applying XGBoost


In [ ]:
import xgboost as xgb
import numpy as np # Import numpy for nan check
from sklearn.metrics import accuracy_score, classification_report

binary_mapping = {
    'TRUE': 0,
    'fake': 1
}

y_train_binary = y_train.map(binary_mapping)
y_test_binary = y_test.map(binary_mapping)


train_valid_indices = ~y_train_binary.isna()
X_train_tfidf = X_train_tfidf[train_valid_indices.to_numpy()]
y_train_binary = y_train_binary[train_valid_indices].astype(int)

test_valid_indices = ~y_test_binary.isna()
X_test_tfidf = X_test_tfidf[test_valid_indices.to_numpy()]
y_test_binary = y_test_binary[test_valid_indices].astype(int)

print("Training Data Shape:", X_train_tfidf.shape)
print("Testing Data Shape:", X_test_tfidf.shape)

xgb_binary_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

print("Training XGBoost (Binary Classification: Real vs Fake)...")
xgb_binary_model.fit(X_train_tfidf, y_train_binary)

y_pred_binary = xgb_binary_model.predict(X_test_tfidf)
y_proba_binary = xgb_binary_model.predict_proba(X_test_tfidf)[:, 1]

print(f"\nBinary Accuracy: {accuracy_score(y_test_binary, y_pred_binary):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_binary, y_pred_binary, target_names=['Real (0)', 'Fake (1)']))


Confusion matrix & Curve (ROC, AUC) for XGBoost

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc


plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
cm_binary = confusion_matrix(y_test_binary, y_pred_binary)
sns.heatmap(cm_binary, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Real (0)', 'Fake (1)'],
            yticklabels=['Real (0)', 'Fake (1)'])
plt.title('Binary Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 2, 2)
fpr_bin, tpr_bin, _ = roc_curve(y_test_binary, y_proba_binary)
roc_auc_bin = auc(fpr_bin, tpr_bin)

plt.plot(fpr_bin, tpr_bin, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc_bin:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('Real Positive Rate (TPR)')
plt.title('Binary ROC Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final AUC score is: {roc_auc_bin:.4f}")

Applying Ensemble

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

clf1 = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf3 = LogisticRegression(max_iter=1000, random_state=42)

ensemble_model = VotingClassifier(
    estimators=[('xgb', clf1), ('rf', clf2), ('lr', clf3)],
    voting='soft'
)

print("Ensemble (XGB + RF + LR)")
ensemble_model.fit(X_train_tfidf, y_train_binary)

y_pred_ensemble = ensemble_model.predict(X_test_tfidf)
y_proba_ensemble = ensemble_model.predict_proba(X_test_tfidf)[:, 1]

print(f"\nEnsemble Accuracy: {accuracy_score(y_test_binary, y_pred_ensemble):.4f}")
print("\nClassification Report (Real vs Fake - Ensemble):")
print(classification_report(y_test_binary, y_pred_ensemble, target_names=['Real (0)', 'Fake (1)']))

Confusion matrix & Curve (ROC, AUC) for Ensemble

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

y_proba_ensemble = ensemble_model.predict_proba(X_test_tfidf)[:, 1]

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
cm_ensemble = confusion_matrix(y_test_binary, y_pred_ensemble)
sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Real (0)', 'Fake (1)'],
            yticklabels=['Real (0)', 'Fake (1)'])
plt.title('Ensemble Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 2, 2)
fpr_ens, tpr_ens, _ = roc_curve(y_test_binary, y_proba_ensemble)
roc_auc_ens = auc(fpr_ens, tpr_ens)

plt.plot(fpr_ens, tpr_ens, color='blue', lw=2, label=f'Ensemble ROC (AUC = {roc_auc_ens:.4f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Ensemble ROC Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Ensemble AUC Score is: {roc_auc_ens:.4f}")

Applying CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score

max_words = 10000
max_len = 200

# Apply the same filtering to X_train and X_test lists
# This ensures that X_train_cnn and X_test_cnn have the same number of samples as y_train_binary and y_test_binary
X_train_cnn = [X_train[i] for i, valid in enumerate(train_valid_indices) if valid]
X_test_cnn = [X_test[i] for i, valid in enumerate(test_valid_indices) if valid]

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_cnn) # Use filtered X_train

X_train_seq = tokenizer.texts_to_sequences(X_train_cnn) # Use filtered X_train
X_test_seq = tokenizer.texts_to_sequences(X_test_cnn)   # Use filtered X_test

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# Verify shapes to ensure they are consistent
print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"y_train_binary shape: {y_train_binary.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")
print(f"y_test_binary shape: {y_test_binary.shape}")

model = Sequential([
    Embedding(max_words, 128, input_length=max_len),

    Conv1D(128, 5, activation='relu'),

    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train_pad, y_train_binary,
                    epochs=5,
                    batch_size=32,
                    validation_split=0.1,
                    verbose=1)

y_pred_prob = model.predict(X_test_pad)
y_pred_cnn = (y_pred_prob > 0.5).astype("int32")

print(f"\nCNN Accuracy: {accuracy_score(y_test_binary, y_pred_cnn):.4f}")
print("\nClassification Report (CNN):")
print(classification_report(y_test_binary, y_pred_cnn, target_names=['Real (0)', 'Fake (1)']))


Confusion matrix & Curve (ROC, AUC) for CNN

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

y_proba_cnn = y_pred_prob.flatten()

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
cm_cnn = confusion_matrix(y_test_binary, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Real (0)', 'Fake (1)'],
            yticklabels=['Real (0)', 'Fake (1)'])
plt.title('CNN Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 2, 2)
fpr_cnn, tpr_cnn, _ = roc_curve(y_test_binary, y_proba_cnn)
roc_auc_cnn = auc(fpr_cnn, tpr_cnn)

plt.plot(fpr_cnn, tpr_cnn, color='darkviolet', lw=2, label=f'CNN ROC (AUC = {roc_auc_cnn:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('CNN ROC Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"CNN AUC Score is: {roc_auc_cnn:.4f}")

Proposed Model Usint CNN and XGBoost

In [ ]:
import numpy as np
import xgboost as xgb
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, accuracy_score

max_words = 10000
max_len = 200

# Filter X_train and X_test to match the filtered y_train_binary and y_test_binary
X_train_filtered_for_cnn = [X_train[i] for i, valid in enumerate(train_valid_indices) if valid]
X_test_filtered_for_cnn = [X_test[i] for i, valid in enumerate(test_valid_indices) if valid]

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_filtered_for_cnn)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_filtered_for_cnn), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_filtered_for_cnn), maxlen=max_len)

input_layer = Input(shape=(max_len,))
embedding = Embedding(max_words, 128)(input_layer)
conv = Conv1D(128, 5, activation='relu')(embedding)
pooling = GlobalMaxPooling1D()(conv)
dense_out = Dense(64, activation='relu')(pooling)
output_layer = Dense(1, activation='sigmoid')(dense_out)

cnn_extractor = Model(inputs=input_layer, outputs=pooling)
full_cnn_model = Model(inputs=input_layer, outputs=output_layer)

full_cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("CNN Feature Extractor ")
full_cnn_model.fit(X_train_seq, y_train_binary, epochs=3, batch_size=32, verbose=0)

X_train_deep_features = cnn_extractor.predict(X_train_seq)
X_test_deep_features = cnn_extractor.predict(X_test_seq)

proposed_xgb = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    scale_pos_weight=10,
    eval_metric='logloss',
    random_state=42
)

print("Proposed Hybrid Model (CNN-XGB) ")
proposed_xgb.fit(X_train_deep_features, y_train_binary)

y_pred_hybrid = proposed_xgb.predict(X_test_deep_features)


print(f"\nProposed Model Accuracy: {accuracy_score(y_test_binary, y_pred_hybrid):.4f}")
print("\nClassification Report (Proposed Hybrid Model):")
print(classification_report(y_test_binary, y_pred_hybrid, target_names=['Real (0)', 'Fake (1)']))

Confusion matrix & Curve (ROC, AUC) for Proposed Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc


y_proba_hybrid = proposed_xgb.predict_proba(X_test_deep_features)[:, 1]

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
cm_hybrid = confusion_matrix(y_test_binary, y_pred_hybrid)
sns.heatmap(cm_hybrid, annot=True, fmt='d', cmap='YlGnBu', cbar=False,
            xticklabels=['Real (0)', 'Fake (1)'],
            yticklabels=['Real (0)', 'Fake (1)'])
plt.title('Proposed Hybrid Model Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')

plt.subplot(1, 2, 2)
fpr_hyb, tpr_hyb, _ = roc_curve(y_test_binary, y_proba_hybrid)
roc_auc_hyb = auc(fpr_hyb, tpr_hyb)

plt.plot(fpr_hyb, tpr_hyb, color='forestgreen', lw=2, label=f'Hybrid ROC (AUC = {roc_auc_hyb:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Proposed Hybrid Model ROC Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Proposed Hybrid Model AUC Score is: {roc_auc_hyb:.4f}")

Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score

# XGBoost
acc_xgb = accuracy_score(y_test_binary, y_pred_binary)
prec_xgb, rec_xgb, f1_xgb, _ = precision_recall_fscore_support(y_test_binary, y_pred_binary, average='binary')
auc_xgb = roc_auc_score(y_test_binary, y_proba_binary)

# Ensemble
acc_ens = accuracy_score(y_test_binary, y_pred_ensemble)
prec_ens, rec_ens, f1_ens, _ = precision_recall_fscore_support(y_test_binary, y_pred_ensemble, average='binary')
auc_ens = roc_auc_score(y_test_binary, y_proba_ensemble)

# CNN
acc_cnn = accuracy_score(y_test_binary, y_pred_cnn)
prec_cnn, rec_cnn, f1_cnn, _ = precision_recall_fscore_support(y_test_binary, y_pred_cnn, average='binary')
auc_cnn = roc_auc_score(y_test_binary, y_proba_cnn)

# Proposed Hybrid
acc_hyb = accuracy_score(y_test_binary, y_pred_hybrid)
prec_hyb, rec_hyb, f1_hyb, _ = precision_recall_fscore_support(y_test_binary, y_pred_hybrid, average='binary')
auc_hyb = roc_auc_score(y_test_binary, y_proba_hybrid)

# Variable
xgb_results = [acc_xgb, prec_xgb, rec_xgb, f1_xgb, auc_xgb]
ensemble_results = [acc_ens, prec_ens, rec_ens, f1_ens, auc_ens]
cnn_results = [acc_cnn, prec_cnn, rec_cnn, f1_cnn, auc_cnn]
hybrid_results = [acc_hyb, prec_hyb, rec_hyb, f1_hyb, auc_hyb]

# Graph
models = ['XGBoost', 'Ensemble', 'CNN', 'Proposed Hybrid']
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC Score']
data = np.array([xgb_results, ensemble_results, cnn_results, hybrid_results])

x = np.arange(len(metrics))
width = 0.18
fig, ax = plt.subplots(figsize=(14, 8))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f1c40f']

for i, model_name in enumerate(models):
    offset = (i - 1.5) * width
    rects = ax.bar(x + offset, data[i], width, label=model_name, color=colors[i])

    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Scores', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=4)

plt.ylim(0, 1.2)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

Loss Function

In [ ]:
from sklearn.metrics import log_loss

# 1. XGBoost Loss
loss_xgb = log_loss(y_test_binary, y_proba_binary)

# 2. Ensemble Loss
loss_ens = log_loss(y_test_binary, y_proba_ensemble)

# 3. CNN Loss
# We take the last value from the training history
loss_cnn = history.history['val_loss'][-1]

# 4. Proposed Hybrid Model Loss
loss_hyb = log_loss(y_test_binary, y_proba_hybrid)

# --- Visualization ---

loss_models = ['XGBoost', 'Ensemble', 'CNN (Val)', 'Proposed Hybrid']
loss_values = [loss_xgb, loss_ens, loss_cnn, loss_hyb]

plt.figure(figsize=(10, 6))
bars = plt.bar(loss_models, loss_values, color=['#3498db', '#2ecc71', '#e74c3c', '#f1c40f'])

# Adding labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height:.4f}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3),
                 textcoords="offset points",
                 ha='center', va='bottom', fontweight='bold')

plt.title('Comparison of Model Loss (Log Loss / Cross-Entropy)', fontsize=14, fontweight='bold')
plt.ylabel('Loss Value (Lower is Better)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# Print Summary Table
loss_df = pd.DataFrame({
    'Model': loss_models,
    'Log Loss': loss_values
})
print("\nFinal Loss Summary:")
print(loss_df.to_string(index=False))